# WebSec Repair - Qwen2.5-Coder Training on Colab

Fine-tune Qwen2.5-Coder-1.5B for vulnerability classification and repair in 15-30 minutes with free GPU.

**Prerequisites:**
- Google Colab account
- 15GB+ free storage
- (Optional) Hugging Face token for model saving


## Step 1: Check GPU

In [ ]:
!nvidia-smi

Expected: NVIDIA GPU (A100, T4, L4, etc.)

If no GPU: Runtime → Change runtime type → GPU

## Step 2: Clone Repository

In [ ]:
!git clone https://github.com/YOUR_ORG/websec_repair_env.git
%cd websec_repair_env
!pwd && ls -la

## Step 3: Install Dependencies

In [ ]:
!pip install -q -r training_requirements.txt
print("✓ Training dependencies installed")

Expected output: "✓ Training dependencies installed"

This installs:
- torch, transformers, peft
- datasets, accelerate
- bitsandbytes (for 4-bit quantization)

## Step 4: Run Training

In [ ]:
!python training/qlora_train.py \
  --output-dir ./qwen-websec-finetuned \
  --model-id Qwen/Qwen2.5-Coder-1.5B-Instruct \
  --epochs 3 \
  --learning-rate 2e-4 \
  --batch-size 4 \
  --gradient-accumulation-steps 4

Expected time: 15-30 minutes

You'll see:
1. Model loading (1-2 min)
2. Training progress (epochs 1-3)
3. Model saved output

The model will be saved to `./qwen-websec-finetuned/`

## Step 5: Test the Model

In [ ]:
from training.inference import WebSecRepairAgent
import json

# Load the trained model
print("Loading model...")
agent = WebSecRepairAgent("./qwen-websec-finetuned")
print("✓ Model loaded!\n")

# Test on SQL injection task
print("Testing SQL Injection task:")
action = agent.get_action(
    task_id="sqli_login",
    instruction="Fix SQL injection in login form",
    code_snippet='query = f"SELECT * FROM users WHERE id={uid}"',
    scanner_hint="SQL injection detected in query construction"
)
print(json.dumps(action, indent=2))
print()

# Test on XSS task
print("Testing XSS task:")
action = agent.get_action(
    task_id="xss_comments",
    instruction="Fix XSS in comment display",
    code_snippet='return f"<div>{comment}</div>"',
    scanner_hint="Cross-site scripting in comment rendering"
)
print(json.dumps(action, indent=2))

Expected output: JSON with `action_type` and `vulnerability_type` (or `patch_id`)

Example:
```json
{
  "action_type": "classify",
  "vulnerability_type": "sql_injection"
}
```

## Step 6 (Optional): Save to Hugging Face Hub

In [ ]:
# Login to Hugging Face
!huggingface-cli login

Paste your HF token when prompted

In [ ]:
# Push model to hub (replace YOUR_USERNAME)
from peft import PeftModel, AutoPeftModelForCausalLM
from transformers import AutoTokenizer

model = AutoPeftModelForCausalLM.from_pretrained("./qwen-websec-finetuned/adapter")
model.push_to_hub("YOUR_USERNAME/qwen-websec-v1")

## Step 7: Download Model from Colab

In [ ]:
from google.colab import files
import zipfile
import os

# Create a zip of the model
!zip -r qwen-websec-finetuned.zip qwen-websec-finetuned/

# Download
files.download("qwen-websec-finetuned.zip")
print("✓ Model downloaded as qwen-websec-finetuned.zip")

## Done!

Your fine-tuned model is ready. Next steps:

1. **Use locally**: Extract the zip and load with `WebSecRepairAgent`
2. **Push to Hub**: Share with your team via Hugging Face
3. **Integrate with environment**: Use in your RL training loop

See [TRAINING_GUIDE.md](./TRAINING_GUIDE.md) for more details.